# WhisperColab · Notebook UI

Whisper **Large-v3** 음성 인식 모델을 **Google Colab 노트북 내부**에서 실행합니다. 다른 웹사이트를 열지 않고, 파일 선택 → 자동 변환 → 진행률 확인 → TXT 다운로드까지 할 수 있습니다.

> **중요:** 이 노트북은 외부 웹서버, Gradio, TryCloudflare, SSH, 원격 제어를 사용하지 않습니다. 무료 Colab의 **노트북 UI 우회 제한**을 피하는 구조이지만, GPU 할당·세션 시간은 여전히 보장되지 않습니다.

### 사용 순서
1. 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다.
2. **런타임 → 모두 실행**을 클릭합니다.
3. 아래에 표시되는 **파일 선택**에서 음성을 고르면 **자동으로 변환이 시작**됩니다.
4. 처리가 끝나면 **TXT 다운로드**를 누릅니다.

모델: OpenAI Whisper Large-v3 / `faster-whisper` (CUDA GPU `int8_float16`) · OpenAI API 키 불필요



In [ ]:
# STEP 1. GPU 확인 및 설치 (Colab 런타임에서 실행)
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True,
)
if gpu.returncode:
    raise RuntimeError(
        "GPU가 없습니다. Colab 메뉴의 '런타임 → 런타임 유형 변경 → T4 GPU'를 선택하고 다시 실행하세요."
    )
print("✅ 연결된 GPU:", gpu.stdout.strip())

repo_dir = Path("/content/WhisperColab")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/dariring/WhisperColab.git", str(repo_dir)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")],
    check=True,
)
print("✅ 준비 완료. 아래 노트북 UI에서 파일을 선택하세요.")



In [ ]:
# STEP 2. Colab 노트북 내부 패널 (외부 웹서버 / 터널 없음)
import html
import json
import os
import re
import subprocess
import sys
import threading
import uuid
from pathlib import Path

import ipywidgets as widgets
import nvidia.cublas
import nvidia.cudnn
from google.colab import files, output
from IPython.display import display

output.enable_custom_widget_manager()

storage = Path("/content/whispercolab")
input_dir = storage / "uploads"
result_dir = storage / "results"
input_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

# CTranslate2가 요구하는 CUDA 12 + cuDNN 9 라이브러리 경로를
# 모델을 실행할 별도의 Python 프로세스에 전달합니다.
library_directories = [
    str(Path(nvidia.cublas.__path__[0]) / "lib"),
    str(Path(nvidia.cudnn.__path__[0]) / "lib"),
]
worker_env = os.environ.copy()
worker_env["LD_LIBRARY_PATH"] = os.pathsep.join(
    library_directories + ([worker_env["LD_LIBRARY_PATH"]] if worker_env.get("LD_LIBRARY_PATH") else [])
)
worker_env["HF_HUB_DISABLE_TELEMETRY"] = "1"

allowed_extensions = {
    ".mp3", ".wav", ".m4a", ".flac", ".ogg", ".opus",
    ".aac", ".mp4", ".webm", ".wma", ".aiff", ".mkv",
}
max_bytes = 75 * 1024 * 1024
state = {"busy": False, "result": None}

heading = widgets.HTML(
    '<div style="padding:22px 0 9px;color:#25324a;font-family:system-ui">'
    '<div style="font-size:13px;color:#6081b2;font-weight:700;letter-spacing:1px">WHISPERCOLAB · LOCAL NOTEBOOK UI</div>'
    '<h2 style="margin:9px 0;font-size:30px;font-weight:750;letter-spacing:-1px">음성을 텍스트로, 간단하게.</h2>'
    '<p style="color:#63748a">음성 파일을 선택하면 Whisper Large-v3가 자동 변환합니다.</p></div>'
)
language = widgets.Dropdown(
    options=[("🌐 언어 자동 감지", "auto"), ("한국어", "ko"), ("English", "en"),
             ("日本語", "ja"), ("中文", "zh")],
    value="auto", description="언어:",
    style={"description_width": "60px"}, layout=widgets.Layout(width="260px"),
)
vad = widgets.Checkbox(value=True, description="무음 구간 자동 제외", indent=False)
progress = widgets.IntProgress(value=0, min=0, max=100, description="진행률:",
                               style={"description_width": "60px"},
                               layout=widgets.Layout(width="98%"))
status = widgets.HTML(value='<span style="color:#70839b">파일을 선택해 주세요.</span>')
preview = widgets.Textarea(
    value="", placeholder="변환된 텍스트가 여기에 표시됩니다.",
    description="", disabled=False,
    layout=widgets.Layout(width="98%", height="220px"),
)
download_button = widgets.Button(
    description="TXT 다운로드 ↓", disabled=True, button_style="primary",
    layout=widgets.Layout(width="170px"),
)
upload_slot = widgets.VBox()
info = widgets.HTML(
    "<small style='color:#7a899a'>최대 75MB · MP3/WAV/M4A/FLAC/OGG/OPUS/MP4 등<br>"
    "파일은 Colab 런타임에만 저장됩니다. 세션 종료 전 TXT를 다운로드하세요.</small>"
)


def update_status(message, color="#53677f"):
    status.value = (
        f'<span style="font-size:13px;color:{color}">{html.escape(str(message))}</span>'
    )


def on_download(_):
    path = state.get("result")
    if path and Path(path).is_file():
        files.download(str(path))
    else:
        update_status("TXT 파일이 없습니다. 변환부터 실행해 주세요.", "#ba4646")


download_button.on_click(on_download)


def reset_uploader():
    uploader = widgets.FileUpload(
        accept=",".join(sorted(allowed_extensions)), multiple=False,
        description="음성 파일 선택", button_style="",
        layout=widgets.Layout(width="200px"),
    )
    uploader.observe(on_uploaded, names="value")
    upload_slot.children = (uploader,)


def worker_task(input_path, output_path, selected_language, selected_vad):
    recent_log = []
    try:
        command = [
            sys.executable, "-u", str(repo_dir / "whisper_worker.py"),
            "--input", str(input_path), "--output", str(output_path),
            "--language", selected_language,
        ]
        if not selected_vad:
            command.append("--no-vad")
        process = subprocess.Popen(
            command, env=worker_env, cwd=str(repo_dir), text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
        )
        detected_language = None
        assert process.stdout is not None
        for line in process.stdout:
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                recent_log.append(line.strip())
                recent_log = recent_log[-15:]
                continue
            if event.get("kind") == "status":
                update_status(event.get("message", "작업 중"))
            elif event.get("kind") == "progress":
                progress.value = min(99, max(0, int(event.get("percent", 0))))
                update_status(event.get("message", "인식 중"))
            elif event.get("kind") == "done":
                detected_language = event.get("language", "unknown")
            elif event.get("kind") == "error":
                recent_log.append(event.get("message", "알 수 없는 오류"))

        code = process.wait()
        if code != 0 or not output_path.exists():
            raise RuntimeError("; ".join(recent_log[-3:]) or "변환 프로세스가 실패했습니다")

        state["result"] = output_path
        full_text = output_path.read_text(encoding="utf-8")
        truncated = len(full_text) > 20000
        preview.value = full_text[:20000]
        download_button.disabled = False
        progress.value = 100
        label = f"✅ 변환 완료! 언어: {detected_language} · {len(full_text):,}자"
        if truncated:
            label += " (미리보기 20,000자 / 전체 TXT 다운로드 가능)"
        update_status(label, "#27815b")
    except Exception as exc:
        update_status(f"❌ 변환 실패: {exc}", "#b13d3d")
    finally:
        input_path.unlink(missing_ok=True)
        state["busy"] = False
        reset_uploader()


def on_uploaded(change):
    # ipywidgets 8은 tuple, 이전 버전은 dict 형태로 파일 정보를 제공합니다.
    uploaded = change.get("new")
    if not uploaded or state["busy"]:
        return
    try:
        item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
        name = item["name"] if isinstance(item, dict) else item.name
        raw = item["content"] if isinstance(item, dict) else item.content
        extension = Path(name.lower()).suffix
        if extension not in allowed_extensions:
            raise ValueError("지원하지 않는 음성/영상 파일 형식입니다")
        if len(raw) > max_bytes:
            raise ValueError("파일이 75MB를 초과했습니다")
        if not raw:
            raise ValueError("빈 파일입니다")

        job_id = uuid.uuid4().hex
        cleaned_name = re.sub(r"[^\w .()-]", "_", Path(name).stem).strip(" ._")[:75] or "transcript"
        input_path = input_dir / (job_id + extension)
        output_path = result_dir / (cleaned_name + "-" + job_id[:8] + ".txt")
        input_path.write_bytes(bytes(raw))

        state["busy"] = True
        state["result"] = None
        progress.value = 0
        preview.value = ""
        download_button.disabled = True
        for child in upload_slot.children:
            child.disabled = True
        update_status("파일 업로드 완료 · 음성 인식 준비 중")

        threading.Thread(
            target=worker_task,
            args=(input_path, output_path, language.value, vad.value),
            daemon=True,
        ).start()
    except Exception as exc:
        update_status(f"❌ {exc}", "#b13d3d")
        reset_uploader()


reset_uploader()
display(widgets.VBox([
    heading,
    widgets.HBox([language, vad], layout=widgets.Layout(flex_flow="row wrap", gap="14px")),
    upload_slot, info, progress, status,
    widgets.HTML("<b>변환 결과 미리보기</b>"),
    preview, download_button,
], layout=widgets.Layout(max_width="850px", padding="12px", gap="12px")))



### 다운로드가 시작되지 않는 경우

일부 브라우저에서는 위젯에서 다운로드 버튼을 클릭해도 저장 창이 뜨지 않을 수 있습니다. 이때는 아래 셀을 직접 실행하거나, 왼쪽 **파일** 메뉴에서 `/content/whispercolab/results`의 TXT를 다운로드하세요.


In [ ]:
# STEP 3 (선택). 위쪽 다운로드 버튼이 동작하지 않을 때만 실행
from google.colab import files
if state.get("result") and Path(state["result"]).exists():
    files.download(str(state["result"]))
else:
    print("먼저 파일을 업로드하고 변환을 완료해 주세요.")

